<a href="https://colab.research.google.com/github/KaisaridiSofia/leaspy_tutorial/blob/simulate_notebook/notebooks/simulate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# DEV, TO DELETE BEFORE MERGING TO MAIN: clone the fork/branch instead of main (delete the leaspy_tutorial folder to re-pull)
# BRANCH = "main"
# REPO = "https://github.com/aramis-lab/leaspy_tutorial"
BRANCH = "simulate_notebook"
REPO = "https://github.com/KaisaridiSofia/leaspy_tutorial"

import importlib.util, pathlib, subprocess, sys

IN_COLAB = (
    importlib.util.find_spec("google") is not None
    and importlib.util.find_spec("google.colab") is not None
)
pip = [sys.executable, "-m", "pip", "install", "-q"]

if IN_COLAB:
    if importlib.util.find_spec("leaspy") is None:
        subprocess.run(pip + ["--no-deps",
            "leaspy @ git+https://github.com/aramis-lab/leaspy.git@v2.1.0"], check=True)
        subprocess.run(pip + ["lifelines"], check=True)   # dep Colab may lack
    if not pathlib.Path("leaspy_tutorial").exists():
        subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO], check=True)
    sys.path.insert(0, "leaspy_tutorial")
else:
    for _root in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
        if (_root / "tutolib").is_dir():
            sys.path.insert(0, str(_root)); break

import tutolib as tp
print("tutolib", tp.__version__, "loaded")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

import leaspy
from leaspy.datasets import load_dataset
from leaspy.models import LogisticModel

print(f"leaspy {leaspy.__version__}")

# Simulation

In this practical, we will use a fitted Leaspy model to generate a **new synthetic longitudinal cohort**. We will then check whether the parameters used to generate that cohort can be recovered.

**Target duration: 10 minutes.**

# Data and fitted model

We keep the same three Parkinson's disease scores as in the fit practical. To spend our time on simulation rather than fitting again, we load a model fitted with the same data, seed and two-source configuration used there.

In [ ]:
FEATURES = ["MDS1_total", "SCOPA_total", "MOCA_total"]
SEED = 0

df = load_dataset("parkinson")[FEATURES]
df_train = df.loc[:"GS-160"]
model = LogisticModel.load(str(tp.asset_path("parkinson_3_scores_model.json")))

print(f"Training cohort: {df_train.index.unique('ID').size} subjects")
print(f"Loaded model: {model.dimension} scores, {model.source_dimension} sources")
display(df_train.head())

# Simulate

Leaspy generates a cohort in three steps:

1. sample each new subject's individual parameters ($\tau$, $\xi$ and sources) from the fitted population distributions;
2. generate that subject's visit times from a visit design;
3. evaluate the model at those times and add observation noise.

The result therefore resembles the fitted population, but it contains entirely new subjects.

In [ ]:
tp.runquestion(7)

## Choose the visit design

With `visit_type="random"`, visit times are controlled by the dictionary below. Times are in years.

| Parameter | Meaning |
| --- | --- |
| `patient_number` | Number of subjects to simulate. |
| `visit_type` | Method used to generate visits; `"random"` samples new schedules. |
| `first_visit_mean` | Mean first-visit offset from each subject's $\tau$. |
| `first_visit_std` | Standard deviation of the first-visit offset. |
| `time_follow_up_mean` | Mean follow-up duration. |
| `time_follow_up_std` | Standard deviation of the follow-up duration. |
| `distance_visit_mean` | Mean time between consecutive visits. |
| `distance_visit_std` | Standard deviation of the time between visits. |
| `min_spacing_between_visits` | Minimum allowed spacing; closer visits are removed. |

Here we request 150 subjects, about 8 years of follow-up, and approximately annual visits.

In [ ]:
visit_parameters = {
    "patient_number": 150,
    "visit_type": "random",
    "first_visit_mean": 0.0,
    "first_visit_std": 0.4,
    "time_follow_up_mean": 8.0,
    "time_follow_up_std": 0.5,
    "distance_visit_mean": 1.0,
    "distance_visit_std": 0.15,
    "min_spacing_between_visits": 1 / 365,
}

visit_parameters

In [ ]:
tp.runquestion(8)

## Task: generate the cohort

Call `model.simulate` with the `"simulate"` algorithm, `FEATURES`, `visit_parameters`, and `SEED`. Store the returned result in `simulated`, then convert its data to a dataframe named `df_sim` indexed by `ID` and `TIME`.

In [ ]:
# Complete the task in this cell or reveal and run the solution.
# simulated = ...
# df_sim = ...
tp.solution(6)

## Inspect the result

The result contains both the noisy longitudinal observations and the known individual parameters that generated them. Keeping those ground-truth parameters is what makes recovery checks possible.

In [ ]:
n_subjects = df_sim.index.unique("ID").size
visits_per_subject = df_sim.groupby(level="ID").size()

print(f"{n_subjects} simulated subjects")
print(f"{len(df_sim)} visits in total")
print(
    f"Visits per subject: median {visits_per_subject.median():.0f} "
    f"(range {visits_per_subject.min()}–{visits_per_subject.max()})"
)
display(df_sim.head())
display(simulated.individual_parameters.head())

## Plot a few simulated trajectories

Subjects differ in disease timing, progression speed and spatial shift, while observation noise adds visit-to-visit variability.

In [ ]:
sample_ids = df_sim.index.unique("ID")[:6]
fig, axes = plt.subplots(1, len(FEATURES), figsize=(14, 3.5), sharex=True)

for feature, ax in zip(FEATURES, axes):
    for subject_id in sample_ids:
        subject = df_sim.loc[subject_id]
        ax.plot(subject.index, subject[feature], marker="o", ms=3, alpha=0.7)
    ax.set_title(feature)
    ax.set_xlabel("Age")
    ax.set_ylim(0, 1)

axes[0].set_ylabel("Normalized score")
plt.tight_layout()
plt.show()

# Simulation quality assessment

A visual check is useful but not sufficient. We now follow a shortened version of the recovery pipeline from `illustration_simulation.ipynb`:

1. fit the same model family to the simulated observations;
2. compare the refitted population parameters with the generating parameters;
3. personalize the simulated subjects and compare their estimated $\tau$, $\xi$ and space shifts with the known values.

This is an **illustrative recovery check**, not a formal guarantee: finite samples and a short stochastic fit prevent exact equality.

## Refit the simulated cohort

We first look for a cached simulation model in `notebooks/assets`. If it is absent, we fit and save a 10,000-iteration model; the reference pipeline uses 100,000 iterations.

In [ ]:
SIM_MODEL_PATH = tp.asset_path("parkinson_3_scores_sim_model.json")

if SIM_MODEL_PATH.exists():
    sim_model = LogisticModel.load(str(SIM_MODEL_PATH))
    print("Cached simulation model loaded.")
else:
    sim_model = LogisticModel(name="logistic", source_dimension=2)
    sim_model.fit(
        df_sim,
        "mcmc_saem",
        seed=SEED,
        n_iter=10000,
        progress_bar=True,
    )
    sim_model.save(str(SIM_MODEL_PATH))
    print("Simulation model fitted and saved.")

## Population-level parameter recovery

In [ ]:
rows = []
for parameter in [
    "tau_mean", "tau_std", "xi_std", "noise_std", "log_g_mean", "log_v0_mean"
]:
    generating = model.parameters[parameter].detach().cpu().numpy().ravel()
    refitted = sim_model.parameters[parameter].detach().cpu().numpy().ravel()

    # Report g and v0 on their interpretable (non-logarithmic) scale.
    if parameter in {"log_g_mean", "log_v0_mean"}:
        generating, refitted = np.exp(generating), np.exp(refitted)

    label = {"log_g_mean": "g", "log_v0_mean": "v0"}.get(parameter, parameter)
    features = FEATURES if len(generating) == len(FEATURES) else [""] * len(generating)

    for feature, true_value, estimated_value in zip(features, generating, refitted):
        relative_error = (estimated_value - true_value) / abs(true_value) * 100
        rows.append({
            "Parameter": label,
            "Feature": feature,
            "Generating": round(float(true_value), 4),
            "Refitted": round(float(estimated_value), 4),
            "Relative error (%)": round(float(relative_error), 1),
        })

population_recovery = pd.DataFrame(rows)
display(population_recovery)

Read this table parameter by parameter. With the fixed seed, the location and noise parameters are recovered closely, while the spread of progression speeds (`xi_std`) is less stable. That discrepancy is useful information: a larger cohort or a longer refit would be needed before claiming precise recovery of every population parameter.

## Individual-level parameter recovery

We compare disease timing $\tau_i$, log-speed $\xi_i$ and the feature-specific space shifts $w_{ik}$. Estimated shifts are reconstructed from the personalized sources and the refitted mixing matrix.

In [ ]:
# `simulate` keeps the parameters used to generate each subject: these are our ground truth.
ip_true = simulated.individual_parameters.astype(float)

# Personalize the refitted model to estimate the same parameters from observations only.
ip_est = sim_model.personalize(
    df_sim, "scipy_minimize", seed=SEED, progress_bar=True
).to_dataframe().astype(float)

# Personalization returns latent sources. Convert them into one feature-specific
# space shift per score using the refitted model's mixing matrix.
source_columns = [f"sources_{i}" for i in range(sim_model.source_dimension)]
shift_columns = [f"w_{i}" for i in range(len(FEATURES))]
mixing_matrix = sim_model.state.get_tensor_value("mixing_matrix").cpu().numpy()
ip_est[shift_columns] = ip_est[source_columns].to_numpy() @ mixing_matrix

# Put the ground-truth subjects in the same order as the estimated subjects.
ip_true = ip_true.loc[ip_est.index]


# ICC measures agreement between true and estimated values; 1 is perfect agreement.
def icc(x, y):
    values = np.column_stack([x, y])
    subject_means = values.mean(axis=1)
    # Compare variability between subjects with the remaining estimation error.
    ms_between = 2 * ((subject_means - values.mean()) ** 2).sum() / (len(x) - 1)
    ms_error = ((values - subject_means[:, None]) ** 2).sum() / len(x)
    return (ms_between - ms_error) / (ms_between + ms_error)


# Each tuple contains: dataframe column, plot title, and optional feature name.
parameters = [("tau", r"$\tau_i$", ""), ("xi", r"$\xi_i$", "")] + [
    (f"w_{k}", rf"$w_{{i,{k}}}$", feature)
    for k, feature in enumerate(FEATURES)
]
metrics = []
fig, axes = plt.subplots(2, 3, figsize=(11, 7))

# Compare known and estimated values for every parameter.
for ax, (parameter, title, feature) in zip(axes.ravel(), parameters):
    true_values = ip_true[parameter]
    estimated_values = ip_est[parameter]
    metrics.append({
        "Parameter": "w" if parameter.startswith("w_") else parameter,
        "Feature": feature,
        "ICC(3,1)": round(float(icc(true_values, estimated_values)), 3),
        "Pearson r": round(float(true_values.corr(estimated_values)), 3),
    })

    # Points on the red identity line have been recovered perfectly.
    ax.scatter(true_values, estimated_values, s=18, alpha=0.65)
    ax.axline((0, 0), slope=1, color="red", linestyle="--", linewidth=1)
    ax.set(title=title, xlabel="Generating value", ylabel="Estimated value", aspect="equal")

# Five parameters use five of the six panels in the 2 x 3 grid.
axes.ravel()[-1].set_visible(False)
individual_recovery = pd.DataFrame(metrics)
display(individual_recovery)
plt.tight_layout()
plt.show()

In [ ]:
tp.runquestion(9)